# Klassifikation mit allen Werten

#### Führen Sie mit dem Algorithmus Ihrer Wahl eine Klassifikationsaufgabe auf Ihren Daten durch.
- Durchführung Klassifikation mit XGBoost Algorithmus (XGBClassifier)
- Zielvariable: `JobSat`
- Ursprüngliche Skala 0-10 zu 3 Klassen zusammengefasst
    - Low: 0-3
    - Medium: 4-6
    - High: 7-10
---
#### Teilen Sie dazu zunächst die Daten auf, um Overfitting beim Trainieren des Algorithmus und bei der Parameterauswahl zu vermeiden. Erklären Sie die gewählte Strategie und die Größenverhältnisse.
- **Gewählte Strategie**
  - **Stufe 1 (Modellwahl/Parametertuning)**
    - Parameter ausschließlich auf den Trainingsdaten optimiert –> dafür **3-fache Cross-Validation (cv=3)** im Trainingssplit
  - **Stufe 2 (Finale Bewertung)**
    - nachdem besten Parameter feststehen, wird finale Modell auf separaten Testsplit bewertet
  - durch `stratify=y` wird die Klassenverteilung in den Splits stabil gehalten

- **Konkrete Umsetzung im Code**
  - `train_test_split(..., test_size=0.2, stratify=y, random_state=42)` → **80% Train / 20% Test**
  - `GridSearchCV(..., cv=3)` → Cross-Validation innerhalb der **80% Trainingsdaten**

- **Größenverhältnisse:**
  - **Trainingsdaten/Testdaten:** 80% / 20%
---
#### Wählen Sie geeignete Features aus und setzen Sie die Parameter des Algorithmus. Beschreiben Sie das gewälhte Vorgehen für die Auswahl der Features und Parameter. Berichten Sie den Parameterraum und die final gewählten Parameter. Geben Sie die Performanz auf den Trainingsdaten (bzw. Entwicklungsdaten, falls verwendet) an.

- **Features**
  - Text: alle `object`-Spalten → zu `__text__` zusammengefügt → TF-IDF
  - Numerik: alle `int/float`-Spalten (bool-OneHot vorher zu 0/1), ohne `JobSat`
- **Vorgehen**
  - Pipeline mit `ColumnTransformer` (TF-IDF für Text, `StandardScaler` für Numerik)
  - Feature-Selektion: `SelectFromModel(XGBClassifier)` (entfernt unwichtige Features)
  - Parameterwahl: `GridSearchCV` mit `cv=3` auf Trainingssplit
- **Parameterraum**
    - TF-IDF:
        - ngram_range: (1,1), (1,2)
        - min_df: 2
        - max_df: 0.9
    - XGBoost:
        - n_estimators: 300, 500
        - max_depth: 4, 6
        - learning_rate: 0.05, 0.1
        - reg_lambda: 1.0, 2.0
        - subsample: 0.8
        - colsample_bytree: 0.8
- **Finale Parameter** Ausgabe aus `grid.best_params_`
---
#### Evaluieren Sie die Klassifikation auf den ungesehenen Testdaten. Betrachten Sie Precision und Recall sowie den F-Wert. Welches Maß ist für Ihre Anwendung wichtiger? Bewerten Sie Ihr Ergebnis. Ist es in der Praxis voraussichtlich zufriedenstellend?


- prüft, in welchem Ordner Notebook sich befindet und ob in dem Ordner Dateien liegen, die wie xgboost* heißen
- lediglich fürs Debugging

In [1]:
import os, glob
print("cwd:", os.getcwd())
print("local xgboost candidates:", glob.glob("xgboost*"))


cwd: C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project
local xgboost candidates: []


- Deinstallation bestehender XG-Boost Installation und anschließende Neuinstallation

In [2]:
import sys
!{sys.executable} -m pip -V
!{sys.executable} -m pip uninstall -y xgboost
!{sys.executable} -m pip install --no-cache-dir -U xgboost


pip 25.2 from C:\Users\MoritzSchwarz\PycharmProjects\data-analytics-project\.venv\Lib\site-packages\pip (python 3.13)



   ---------------------------------------- 0.0/72.0 MB ? eta -:--:--
    --------------------------------------- 1.3/72.0 MB 10.0 MB/s eta 0:00:08
   -- ------------------------------------- 3.7/72.0 MB 11.1 MB/s eta 0:00:07
   --- ------------------------------------ 6.3/72.0 MB 11.6 MB/s eta 0:00:06
   ----- ---------------------------------- 9.2/72.0 MB 12.0 MB/s eta 0:00:06
   ------ --------------------------------- 11.8/72.0 MB 12.1 MB/s eta 0:00:05
   -------- ------------------------------- 14.4/72.0 MB 12.2 MB/s eta 0:00:05
   --------- ------------------------------ 17.3/72.0 MB 12.4 MB/s eta 0:00:05
   ---------- ----------------------------- 19.7/72.0 MB 12.3 MB/s eta 0:00:05
   ------------ --------------------------- 22.0/72.0 MB 12.2 MB/s eta 0:00:05
   ------------- -------------------------- 24.6/72.0 MB 12.2 MB/s eta 0:00:04
   --------------- ------------------------ 27.5/72.0 MB 12.2 MB/s eta 0:00:04
   ---------------- ----------------------- 29.6/72.0 MB 12.1 MB/


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix


## Daten laden

Im One-Hot-Encoded Datensatz sind die One-Hot-Encoded Spalten mit bool Werten. Um es etwas einfacher zu gestalten werden diese Werte hier in Integer-Werte (False -> 0; True -> 1) umgewandelt.

In [26]:
df = pd.read_csv("One-Hot-Encoded.csv")

bool_cols = df.select_dtypes(include=['bool']).columns

for col in bool_cols:
    df[col] = df[col].astype(int)
df.dtypes

MaxAge                                                            float64
AgeNum                                                            float64
EdLevel                                                           float64
WorkExp                                                           float64
YearsCode                                                         float64
                                                                   ...   
AIAgents_no, i use ai exclusively in copilot/autocomplete mode      int64
AIAgents_yes, i use ai agents at work daily                         int64
AIAgents_yes, i use ai agents at work monthly or infrequently       int64
AIAgents_yes, i use ai agents at work weekly                        int64
AIAgents_nan                                                        int64
Length: 544, dtype: object

## Zielvariable in Klassen einteilen 

Dadurch haben wir mehr Trainingsdaten, als wenn wir `JobSat` von 0-10 als Klassen definieren würden, `NaN`-Werte werden zuvor gedroppt

Klassen
- **Low**: 0–3
- **Medium**: 4–6
- **High**: 7–10


In [27]:
# Nur Zeilen behalten, wo JobSat vorhanden ist
df = df.dropna(subset=["JobSat"]).copy()

def map_jobsat(x):
    x = float(x)
    if x <= 3:
        return 0
    elif x <= 6:
        return 1
    else:
        return 2

## Feature-Spalten bestimmen

- Textspalten: `object` (Strings)
- Numerische Spalten: `int/float`

### Warum?
- wir wollen Textinfos und numerische Informationen gemeinsam nutzen

#### Aufpassen
Zielvariable darf nicht als Feature verwendet werden, da das Modell sonst die "Lösungen" kennt, wird dementsprechend hier gedroppt: `df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])`

In [28]:
# Textspalten (Strings)
text_cols = df.select_dtypes(include=["object"]).columns.tolist()

df["__text__"] = df[text_cols].fillna("").agg(" ".join, axis=1)

# Numerische Spalten
num_cols = df.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()

# Zielspalte aus numerischen Features entfernen (falls vorhanden)
num_cols = [c for c in num_cols if c != "JobSat"]

df = df.dropna(subset=["__text__"] + num_cols + ["JobSat"])

y = df["JobSat"].apply(map_jobsat)

print("Textspalten:", text_cols)
print("Numerische Spalten:", num_cols)

# Feature-Matrix aus den ausgewählten Spalten
X = df[["__text__"] + num_cols].copy()

X.head()

Textspalten: ['LanguageWantToWorkWith', 'DatabaseWantToWorkWith', 'PlatformWantToWorkWith', 'WebframeWantToWorkWith', 'DevEnvsWantToWorkWith', 'OfficeStackAsyncWantToWorkWith', 'AIModelsWantToWorkWith', 'AIAgent_Uses']
Numerische Spalten: ['MaxAge', 'AgeNum', 'EdLevel', 'WorkExp', 'YearsCode', 'OrgSize', 'RemoteCategoryNum', 'CompTotal', 'ConvertedCompYearly', 'ConvertedCompTotal', 'LanguageHaveWorkedWith__none', "LanguageHaveWorkedWith__['bash/shell (all shells)', 'python', 'sql']", "LanguageHaveWorkedWith__['c#', 'html/css', 'javascript', 'powershell', 'sql', 'typescript']", "LanguageHaveWorkedWith__['c#', 'html/css', 'javascript', 'sql', 'typescript']", "LanguageHaveWorkedWith__['c#', 'html/css', 'javascript', 'sql']", "LanguageHaveWorkedWith__['c#']", "LanguageHaveWorkedWith__['html/css', 'javascript', 'php', 'sql']", "LanguageHaveWorkedWith__['html/css', 'javascript', 'typescript']", "LanguageHaveWorkedWith__['python', 'sql']", "LanguageHaveWorkedWith__['python']", 'LanguageHaveWo

,__text__,MaxAge,AgeNum,EdLevel,WorkExp,YearsCode,OrgSize,RemoteCategoryNum,CompTotal,ConvertedCompYearly,...,"AISelect_yes, i use ai tools monthly or infrequently","AISelect_yes, i use ai tools weekly",AISelect_nan,"AIAgents_no, and i don't plan to","AIAgents_no, but i plan to","AIAgents_no, i use ai exclusively in copilot/autocomplete mode","AIAgents_yes, i use ai agents at work daily","AIAgents_yes, i use ai agents at work monthly or infrequently","AIAgents_yes, i use ai agents at work weekly",AIAgents_nan
0,['dart'] [] [] [] [] ['markdown file'] [] ['so...,34.0,29.0,0.833333,8.0,14.0,0.285714,0.00,52800.0,61256.0,...,1,0,0,0,0,0,0,1,0,0
1,"['java', 'python', 'swift'] ['dynamodb', 'mong...",34.0,29.0,0.500000,2.0,10.0,0.571429,0.25,90000.0,104413.0,...,0,1,0,1,0,0,0,0,0,0
3,"['java', 'kotlin'] [] ['amazon web services (a...",44.0,39.0,0.666667,4.0,5.0,1.000000,0.00,31200.0,36197.0,...,0,1,0,0,0,0,0,1,0,0
7,"['assembly', 'bash/shell (all shells)', 'html/...",44.0,39.0,1.000000,22.0,30.0,0.142857,0.00,72000.0,72000.0,...,0,0,0,0,1,0,0,0,0,0
8,"['scala'] ['dynamodb', 'mysql', 'postgresql'] ...",34.0,29.0,0.666667,9.0,15.0,0.857143,0.00,70000.0,70000.0,...,1,0,0,0,0,0,0,0,0,1


## Train/Test Split
- Aufteilen der Daten in 80% Training und 20% Test
- `stratify=y` sorgt für ungefähre Gleichverteilung der Klassen in Test und Trainingsdaten


In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Train class distribution:\n", y_train.value_counts(normalize=True))
print("Test class distribution:\n", y_test.value_counts(normalize=True))

Train size: 12245
Test size: 3062
Train class distribution:
 JobSat
2    0.719722
1    0.219028
0    0.061249
Name: proportion, dtype: float64
Test class distribution:
 JobSat
2    0.719464
1    0.219138
0    0.061398
Name: proportion, dtype: float64


## Preprocessing für Text und numerische Spalten
- je nach Spaltentyp andere Vorverarbeitung
    - TF-IDF macht aus Text numerische Features
    - StandardScaler sakliert Werte, damit Größeordnungen vergleichbar sind

In [30]:
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "__text__"),
        ("num", StandardScaler(), num_cols)
    ],
    remainder="drop"
)

## Pipeline definieren

- Preprocessing
- Feature-Selektion (SelectFromModel mit XGBoost-Feature-Importances, wirft weniger wichtige Features raus)
- Klassifikator (XGBClassifier, trainiert auf ausgewählten Features)
- Gründstäzlich automatische Reduktion der Featuremenge auf die wichtigsten, damit das Modell besser generalisiert und so schneller trainiert


In [31]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("feature_selection", SelectFromModel(XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ))),
    ("classifier", XGBClassifier(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

## GridSearchCV
- definiert Suchraum für Parameter
- GridSearch testet Kombinationen der Parameter systematisch und wählt die beste Kombination per Cross-Validation (`cv`)
- `cv=3` bedeutet, dass eine 3-fache Validierung auf dem Trainingsset vorgenommen wurde


In [32]:
parameters = {
    # TF-IDF: nur word, nur die zwei wichtigsten Varianten
    "preprocessing__text__analyzer": ["word"],
    "preprocessing__text__ngram_range": [(1, 1), (1, 2)],
    "preprocessing__text__max_df": [0.9],
    "preprocessing__text__min_df": [2],

    # XGBoost: kompakter, sinnvoller Suchraum
    "classifier__n_estimators": [300, 500],
    "classifier__max_depth": [4, 6],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__subsample": [0.8],
    "classifier__colsample_bytree": [0.8],
    "classifier__reg_lambda": [1.0, 2.0],
}

grid = GridSearchCV(pipeline, param_grid=parameters, verbose=2, cv=3, n_jobs=-1)

## Grid Search + Beste Parameter
- Viele Modellvarianten werden trainiert und die beste Variante gefunden
- Output ist dann der beste CV-Score mit den dazugehörigen Parametern

In [33]:
grid.fit(X_train, y_train)

print("Beste Performance:", grid.best_score_)
print("Beste Parameter:\n", grid.best_params_) # Wird für Beantwortung für Mindestanforderung Frage 3 noch benötigt

Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=1.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=0.9, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=  16.7s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=1.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=0.9, preprocessing__text__min_df=2, preprocessing__text__ngram_range=(1, 1); total time=  16.7s
[CV] END classifier__colsample_bytree=0.8, classifier__learning_rate=0.05, classifier__max_depth=4, classifier__n_estimators=300, classifier__reg_lambda=2.0, classifier__subsample=0.8, preprocessing__text__analyzer=word, preprocessing__text__max_df=

## Evaluation auf Testdaten
- Test-Datensatz Vorhersage samt Metriken
- Am Ende zählt Performance auf Test-Daten, welche Modell noch nicht kennt

In [34]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)

print("Classification Report (Test):")
print(classification_report(y_test, y_pred))

print("Confusion Matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=[0,1,2]))

Classification Report (Test):
              precision    recall  f1-score   support

           0       0.14      0.02      0.03       188
           1       0.40      0.12      0.18       671
           2       0.74      0.96      0.84      2203

    accuracy                           0.72      3062
   macro avg       0.43      0.36      0.35      3062
weighted avg       0.63      0.72      0.64      3062

Confusion Matrix (rows=true, cols=pred):
[[   3   37  148]
 [   9   78  584]
 [  10   79 2114]]


- Ausgabe gleicher Metriken für Trainingsdaten um Test- und Trainingsdaten zu vergleichen
- Over- / Underfitting?

In [35]:
# Muss noch ausgeführt werden, um Mindestanforderung Frage 3 zu beantworten

y_pred_train = best_model.predict(X_train)

print("Classification Report (Train):")
print(classification_report(y_train, y_pred_train))

Classification Report (Train):
              precision    recall  f1-score   support

           0       0.99      0.45      0.62       750
           1       0.93      0.43      0.58      2682
           2       0.82      1.00      0.90      8813

    accuracy                           0.84     12245
   macro avg       0.92      0.62      0.70     12245
weighted avg       0.86      0.84      0.81     12245

